In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2
import os
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchinfo

from Generator import generate_image, noise_add, noise_mul
from Model import EdgePreservingDenoiser, SyntheticNoiseDataset, evaluate_model
from Helper import load_checkpoint_generic, save_checkpoint_generic


In [2]:
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_devices}\n")

    for i in range(num_devices):
        device_name = torch.cuda.get_device_name(i)
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9  # in GB
        print(f"Device {i}: {device_name} ({total_memory:.2f} GB)")

Number of CUDA devices: 1

Device 0: NVIDIA GeForce MX150 (4.29 GB)


In [3]:
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [4]:
# === HYPERPARAMETRI ===
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_ITERATIONS = 100000  # 1000 za testiranje, 100000 za full training
VALIDATION_INTERVAL = 100  # Evalvacija vsakih N iteracij
CHECKPOINT_INTERVAL = 100  # Shranjevanje checkpointa
LOG_INTERVAL = 20  # Log vsakih N iteracij

# Dataset
TRAIN_SIZE = 1000  # Train dataset velikost
VAL_SIZE = 200  # Validation dataset velikost

# Noise type
# NOISE_FN = noise_add
NOISE_FN = noise_mul
NOISE_RANGE = (0.1, 0.3)

# Loss function
LOSS_FN = 'MSE'  # MSE, RMSE, MAE

# Checkpoint directory
CHECKPOINT_DIR = f"checkpoints-{NOISE_FN.__name__}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [5]:
from pathlib import Path
import pickle


def load_dataset_arrays(checkpoint_dir, dataset_name):
    """
    Naloži dataset arrays iz pickle file.

    Args:
        checkpoint_dir: Direktorij s checkpointi
        dataset_name: Ime dataseta ('train' ali 'val')

    Returns:
        (clean_images, noisy_images) ali (None, None) če ne obstaja
    """
    checkpoint_dir = Path(checkpoint_dir)
    dataset_path = checkpoint_dir / f"{dataset_name}_dataset.pkl"

    if dataset_path.exists():
        print(f"📂 Loading {dataset_name} dataset from {dataset_path}")
        with open(dataset_path, 'rb') as f:
            data = pickle.load(f)

        clean_images = data['clean']
        noisy_images = data['noisy']
        print(f"✅ Loaded {len(clean_images)} samples")
        return clean_images, noisy_images
    else:
        print(f"⚠️ No {dataset_name} dataset found at {dataset_path}")
        return None, None


def save_dataset_arrays(checkpoint_dir, dataset_name, dataset, overwrite=False):
    """
    Shrani dataset arrays v pickle file direktno iz SyntheticNoiseDataset objekta.

    Args:
        checkpoint_dir: Direktorij za shranjevanje
        dataset_name: Ime dataseta ('train' ali 'val')
        dataset: SyntheticNoiseDataset objekt
        overwrite: Če True, prepiše obstoječ file
    """
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True)
    dataset_path = checkpoint_dir / f"{dataset_name}_dataset.pkl"

    if dataset_path.exists() and not overwrite:
        print(f"⚠️ Dataset already exists at {dataset_path}. Set overwrite=True to replace.")
        return

    # Shrani kot torch tensors v CHW formatu
    data = {
        'clean': torch.stack(dataset.clean),  # Shape: [N, C, H, W]
        'noisy': torch.stack(dataset.noisy)  # Shape: [N, C, H, W]
    }

    print(f"💾 Saving {dataset_name} dataset to {dataset_path}")
    with open(dataset_path, 'wb') as f:
        pickle.dump(data, f)

    print(f"✅ Saved {len(dataset)} samples")

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

# Model
model = EdgePreservingDenoiser().to(device)

# === LOAD OR GENERATE TRAIN DATASET ===
train_clean, train_noisy = load_dataset_arrays(CHECKPOINT_DIR, "train")

if train_clean is not None:
    # Podatki obstajajo - uporabi jih (že v CHW formatu)
    train_dataset = SyntheticNoiseDataset(clean_images=train_clean, noisy_images=train_noisy)
    print(f"✅ Train dataset created from loaded data: {len(train_dataset)} samples")
else:
    # Podatki ne obstajajo - generiraj nove
    print(f"🎨 Generating train dataset ({TRAIN_SIZE} samples)...")
    train_dataset = SyntheticNoiseDataset(
        num_samples=TRAIN_SIZE,
        noise_fn=NOISE_FN,
        noise_range=NOISE_RANGE
    )
    # Shrani dataset direktno (že v CHW formatu)
    save_dataset_arrays(CHECKPOINT_DIR, "train", train_dataset)
    print(f"✅ Train dataset generated and saved: {len(train_dataset)} samples")

# === LOAD OR GENERATE VALIDATION DATASET ===
val_clean, val_noisy = load_dataset_arrays(CHECKPOINT_DIR, "val")

if val_clean is not None:
    # Podatki obstajajo - uporabi jih (že v CHW formatu)
    val_dataset = SyntheticNoiseDataset(clean_images=val_clean, noisy_images=val_noisy)
    print(f"✅ Validation dataset created from loaded data: {len(val_dataset)} samples")
else:
    # Podatki ne obstajajo - generiraj nove
    print(f"🎨 Generating validation dataset ({VAL_SIZE} samples)...")
    val_dataset = SyntheticNoiseDataset(
        num_samples=VAL_SIZE,
        noise_fn=NOISE_FN,
        noise_range=NOISE_RANGE
    )
    # Shrani dataset direktno (že v CHW formatu)
    save_dataset_arrays(CHECKPOINT_DIR, "val", val_dataset)
    print(f"✅ Validation dataset generated and saved: {len(val_dataset)} samples")

# === CREATE DATALOADERS ===
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Loss function
if LOSS_FN == 'MSE':
    criterion = nn.MSELoss()
elif LOSS_FN == 'RMSE':
    criterion = lambda pred, target: torch.sqrt(nn.MSELoss()(pred, target))
elif LOSS_FN == 'MAE':
    criterion = nn.L1Loss()
else:
    raise ValueError(f"Unknown loss function: {LOSS_FN}")

print(f"✅ Loss function: {LOSS_FN}")

Device:  cuda
⚠️ No train dataset found at checkpoints-noise_mul\train_dataset.pkl
🎨 Generating train dataset (1000 samples)...
💾 Saving train dataset to checkpoints-noise_mul\train_dataset.pkl
✅ Saved 1000 samples
✅ Train dataset generated and saved: 1000 samples
⚠️ No val dataset found at checkpoints-noise_mul\val_dataset.pkl
🎨 Generating validation dataset (200 samples)...
💾 Saving val dataset to checkpoints-noise_mul\val_dataset.pkl
✅ Saved 200 samples
✅ Validation dataset generated and saved: 200 samples
✅ Optimizer: AdamW (lr=0.001)
✅ Loss function: MSE


Layer (type:depth-idx)                   Output Shape              Param #
EdgePreservingDenoiser                   [1, 3, 256, 256]          --
├─FilterBranch: 1-1                      [1, 3, 8, 256, 256]       --
│    └─Conv2d: 2-1                       [1, 8, 256, 256]          976
│    └─Conv2d: 2-2                       [1, 8, 256, 256]          976
│    └─Conv2d: 2-3                       [1, 8, 256, 256]          976
├─WeightBranch: 1-2                      [1, 8, 256, 256]          --
│    └─ResNetBlock: 2-4                  [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                  [1, 32, 256, 256]         128
│    │    └─Conv2d: 3-2                  [1, 32, 256, 256]         896
│    │    └─Dropout2d: 3-3               [1, 32, 256, 256]         --
│    │    └─ReLU: 3-4                    [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-5                  [1, 32, 256, 256]         9,248
│    │    └─Dropout2d: 3-6               [1, 32, 256, 256]         --
│    │ 

In [ ]:
torchinfo.summary(model, input_size=(1, 3, 256, 256), device=device)

In [7]:
# Load checkpoint
checkpoint = load_checkpoint_generic(CHECKPOINT_DIR, device=device)

start_iteration = 1
train_losses = []
val_losses = []
val_snrs = []
val_psnrs = []

if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_iteration = checkpoint.get('iteration', 0) + 1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    val_snrs = checkpoint.get('val_snrs', [])
    val_psnrs = checkpoint.get('val_psnrs', [])
    print(f"🔄 Resuming from iteration {start_iteration}")
else:
    print("🚀 Starting training from scratch")

model.train()

# Iterator za neskončno iteracijo skozi dataset
train_iterator = iter(train_loader)

pbar = tqdm(range(start_iteration, MAX_ITERATIONS + 1), initial=start_iteration - 1, total=MAX_ITERATIONS)

for iteration in pbar:
    try:
        noisy, clean = next(train_iterator)
    except StopIteration:
        # Restart iterator če pridemo do konca dataseta
        train_iterator = iter(train_loader)
        noisy, clean = next(train_iterator)

    noisy = noisy.to(device)
    clean = clean.to(device)

    # Forward pass
    optimizer.zero_grad()
    denoised = model(noisy)
    loss = criterion(denoised, clean)

    # Backward pass
    loss.backward()
    optimizer.step()

    # Log training loss
    train_losses.append(loss.item())

    # Update progress bar
    pbar.set_postfix({'loss': f'{loss.item():.6f}'})

    # === VALIDATION ===
    if iteration % VALIDATION_INTERVAL == 0:
        val_loss, val_snr, val_psnr = evaluate_model(model, val_loader, device)
        val_losses.append(val_loss)
        val_snrs.append(val_snr)
        val_psnrs.append(val_psnr)

        print(f"\n📊 Iteration {iteration}/{MAX_ITERATIONS}")
        print(f"   Train Loss: {loss.item():.6f}")
        print(f"   Val Loss: {val_loss:.6f}")
        print(f"   Val SNR: {val_snr:.2f} dB")
        print(f"   Val PSNR: {val_psnr:.2f} dB")

    # === CHECKPOINT SAVING ===
    if iteration % CHECKPOINT_INTERVAL == 0 or iteration == MAX_ITERATIONS:
        save_checkpoint_generic(
            CHECKPOINT_DIR,
            iteration,
            {
                'iteration': iteration,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_losses': train_losses,
                'val_losses': val_losses,
                'val_snrs': val_snrs,
                'val_psnrs': val_psnrs,
                'config': {
                    'batch_size': BATCH_SIZE,
                    'learning_rate': LEARNING_RATE,
                    'noise_fn': NOISE_FN.__name__,
                    'loss_fn': LOSS_FN
                }
            },
            max_checkpoints=4
        )

final_model_path = f"denoiser_{NOISE_FN.__name__}_iter{MAX_ITERATIONS}.pth"
torch.save(model.state_dict(), final_model_path)
print(f"💾 Final model saved: {final_model_path}")

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)

✅ Loaded checkpoint: checkpoints-noise_mul\checkpoint_iteration_500.pth (iteration 500)
🔄 Resuming from iteration 501


  1%|          | 600/100000 [03:43<100:35:52,  3.64s/it, loss=20517.632812]


📊 Iteration 600/100000
   Train Loss: 20517.632812
   Val Loss: 21225.146484
   Val SNR: 13.70 dB
   Val PSNR: -38.56 dB
💾 Checkpoint saved: checkpoints-noise_mul\checkpoint_iteration_600.pth
🗑️  Removed old checkpoint: checkpoints-noise_mul\checkpoint_iteration_200.pth


  1%|          | 700/100000 [07:35<106:50:29,  3.87s/it, loss=20950.656250]


📊 Iteration 700/100000
   Train Loss: 20950.656250
   Val Loss: 21225.146484
   Val SNR: 13.70 dB
   Val PSNR: -38.56 dB
💾 Checkpoint saved: checkpoints-noise_mul\checkpoint_iteration_700.pth
🗑️  Removed old checkpoint: checkpoints-noise_mul\checkpoint_iteration_300.pth


...


📊 Iteration 900/100000
   Train Loss: 21070.183594
   Val Loss: 21225.145926
   Val SNR: 13.70 dB
   Val PSNR: -38.56 dB
💾 Checkpoint saved: checkpoints-noise_mul\checkpoint_iteration_900.pth
🗑️  Removed old checkpoint: checkpoints-noise_mul\checkpoint_iteration_500.pth


  1%|          | 1000/100000 [19:06<99:54:15,  3.63s/it, loss=21849.091797]


📊 Iteration 1000/100000
   Train Loss: 21849.091797
   Val Loss: 21225.145647
   Val SNR: 13.70 dB
   Val PSNR: -38.56 dB
💾 Checkpoint saved: checkpoints-noise_mul\checkpoint_iteration_1000.pth
🗑️  Removed old checkpoint: checkpoints-noise_mul\checkpoint_iteration_600.pth


  1%|          | 1010/100000 [19:30<63:07:47,  2.30s/it, loss=21059.958984]


KeyboardInterrupt: 